In [1]:
#
import requests
import pandas as pd 
pd.set_option('display.max_rows', None)
from IPython.display import HTML

#
import mysql.connector
from mysql.connector import Error

#
import numpy as np

#
import os 
from dotenv import load_dotenv
load_dotenv()
password_sql = os.getenv("PASS_SQL")

import json

# EXTRACCIÓN DE DATOS EN DEEZER

In [100]:
# Usamos la función para extraer todos los cantantes a un CSV 
def extraccion_deezer(endpoint):

    try:
        # 
        
        datos_musica = requests.get(endpoint)
        if datos_musica.status_code == 200:
            print ("API connected")
            # 
            df_musica = pd.DataFrame([datos_musica.json()])
            return df_musica
        else:
            # 
            print ("API failed")

    #         
    except requests.exceptions.ConnectionError as CnxE:
        print (CnxE)

    # 
    except requests.exceptions.Timeout as TO:
        print (TO)
    
    # 
    except requests.exceptions.RequestException as e:
        print (e) 

In [94]:
# Añadimos a una lista los ID de los cantantes o grupos que queremos extraer
artist_id = [12246, 160, 145, 564, 75491, 75798, 290, 483, 10803980, 1538640, 892, 10583405, 412, 13, 4050205, 384236, 119, 5620251, 259, 5962948, 196, 485, 425, 315929, 2446, 1755, 180, 98, 10977, 1434]

def extraer_artistas(artist_id):

    lista_artistas = []
    for artist in artist_id:
        endpoint = f"https://api.deezer.com/artist/{artist}"

        df_artistas = extraccion_deezer(endpoint)
        df_artistas = df_artistas[["id", "name"]]
        lista_artistas.append(df_artistas)
    df_final = pd.concat(lista_artistas)
    return df_final

In [95]:
df_final = extraer_artistas(artist_id)

API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected


In [5]:
df_final

,id,name
0,12246,Taylor Swift
0,160,Shakira
0,145,Beyoncé
0,564,Rihanna
0,75491,Lady Gaga
0,75798,Adele
0,290,Madonna
0,483,Britney Spears
0,10803980,BLACKPINK
0,1538640,Little Mix


In [101]:
##ESTE HAY QUE MODIFICARLO

def extraer_canciones(artist_id):

    lista_canciones_totales = []
    for artist in artist_id:
        endpoint = f"https://api.deezer.com/artist/{artist}/top?limit=50"
        canciones = extraccion_deezer(endpoint)
        lista_canciones_totales.extend(canciones["data"][0])
        
    lista_canciones = []
    for cancion in lista_canciones_totales:
        id_cancion = cancion["id"]
        titulo = cancion["title"]
        duracion = cancion["duration"]
        rank = cancion["rank"]
        id_album = cancion["album"]["id"]
        if len(cancion["contributors"]) == 1:
            colaboradores = False
        else:
            colaboradores = True


        diccionario_cancion = {
                "id": id_cancion,
                "title": titulo,
                "duration": duracion,
                "rank": rank,
                "album_id": id_album,
                "contributors_bool": colaboradores
            }         
                        
        #aqui hay qye nmeter lo de data, y ver que columnas queremos sacar
        lista_canciones.append(diccionario_cancion)
        
    df_final = pd.DataFrame(lista_canciones)
    return df_final, lista_canciones

In [102]:
df_canciones ,  mi_lista = extraer_canciones(artist_id)  ## IDEA SERIA QUE CADA
df_canciones.to_csv("listado_de_canciones.csv", index=False)



API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected


In [8]:
def conteo_artistas(artist_id):

    diccionario_artistas = {}
    for artist in artist_id:
        response = requests.get(f"https://api.deezer.com/artist/{artist}/top?limit=50")
        diccionario_artistas[artist] = len(response.json()["data"])
    return diccionario_artistas

In [9]:
diccionario_artistas = conteo_artistas(artist_id)
diccionario_artistas

{12246: 50,
 160: 50,
 145: 50,
 564: 50,
 75491: 50,
 75798: 50,
 290: 50,
 483: 50,
 10803980: 50,
 1538640: 50,
 892: 50,
 10583405: 50,
 412: 50,
 13: 50,
 4050205: 50,
 384236: 50,
 119: 50,
 5620251: 50,
 259: 50,
 5962948: 50,
 196: 50,
 485: 50,
 425: 50,
 315929: 50,
 2446: 50,
 1755: 50,
 180: 50,
 98: 50,
 10977: 50,
 1434: 50}

In [21]:
endpoint_genero = "https://api.deezer.com/genre/"
def extraer_genero(endpoint_genero):

    try:
        df_genero = extraccion_deezer(endpoint_genero)
        df_data = pd.DataFrame(df_genero["data"][0])
        df_final = df_data[["id","name"]]                       
        return df_final
    except:
        print ("error")

In [23]:
df_genero = extraer_genero(endpoint_genero)
HTML(df_genero.to_html(index=False))

API connected


id,name
0,Todos
132,Pop
116,Rap/Hip Hop
122,Reggaeton
152,Rock
113,Dance
165,R&B
85,Alternativo
106,Electro
466,Folk


In [113]:
def extraer_album():
    album_ids_unicos = set()

    for cancion in mi_lista:
        album_ids_unicos.add(cancion["album_id"])

    lista_albumes = []
    for album_id in album_ids_unicos:
        endpoint = f"https://api.deezer.com/album/{album_id}"
        df_albumes = extraccion_deezer(endpoint)
        df_seleccion = df_albumes[["id", "title", "genre_id", "nb_tracks", "release_date"]]
        lista_albumes.append(df_seleccion)
    
    df_final = pd.concat(lista_albumes)
    return df_final
        


In [114]:
df_albumes = extraer_album()
df_albumes

API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API co

,id,title,genre_id,nb_tracks,release_date
0,14581762,Ride The Lightning (Remastered),464,8,2016-04-15
0,81931,Like a Virgin,132,11,1984-11-14
0,81934,True Blue,132,11,1986-06-11
0,1335314,Talk That Talk (Deluxe),132,14,2011-11-21
0,75700242,Booty,116,1,2018-10-24
0,575252501,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,132,31,2024-04-19
0,928911381,ALGO TÚ,132,1,2026-03-04
0,7534614,Waterloo (Deluxe Edition),132,19,2014-01-01
0,112668,LOVG - Grandes Exitos,132,16,2008-06-18
0,430985247,un x100to,71,1,2023-04-17


In [ ]:
def extraer_albumes():

    lista_albumes = []
    for cancion in mi_lista:
        album_id = cancion["album_id"]
        endpoint = f"https://api.deezer.com/album/{album_id}"
        datos_albumes = extraccion_deezer(endpoint)

        albumes_limpio = {
            "id": datos_albumes.get("id"),
            "title": datos_albumes.get("title"),
            "genre_id": datos_albumes.get("genre_id")
        }

        lista_albumes.append(albumes_limpio)
    df_final = pd.DataFrame(lista_albumes)
    return df_final

In [88]:
df_album = extraer_albumes()

API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API connected
API co

In [89]:
df_album

,id,title,genre_id
0,"0 829966251 Name: id, dtype: int64","0 The Life of a Showgirl Name: title, dtype...","0 132 Name: genre_id, dtype: int64"
1,"0 829966251 Name: id, dtype: int64","0 The Life of a Showgirl Name: title, dtype...","0 132 Name: genre_id, dtype: int64"
2,"0 829966251 Name: id, dtype: int64","0 The Life of a Showgirl Name: title, dtype...","0 132 Name: genre_id, dtype: int64"
3,"0 829966251 Name: id, dtype: int64","0 The Life of a Showgirl Name: title, dtype...","0 132 Name: genre_id, dtype: int64"
4,"0 368474187 Name: id, dtype: int64","0 Midnights Name: title, dtype: str","0 132 Name: genre_id, dtype: int64"
5,"0 9007781 Name: id, dtype: int64","0 1989 (Deluxe) Name: title, dtype: str","0 132 Name: genre_id, dtype: int64"
6,"0 368474187 Name: id, dtype: int64","0 Midnights Name: title, dtype: str","0 132 Name: genre_id, dtype: int64"
7,"0 167766152 Name: id, dtype: int64","0 folklore (deluxe version) Name: title, dt...","0 85 Name: genre_id, dtype: int64"
8,"0 108447472 Name: id, dtype: int64","0 Lover Name: title, dtype: str","0 132 Name: genre_id, dtype: int64"
9,"0 9007781 Name: id, dtype: int64","0 1989 (Deluxe) Name: title, dtype: str","0 132 Name: genre_id, dtype: int64"


In [103]:
df_album.to_csv("listado_de_albumes.csv", index=False)